In [ ]:
# %% Deep learning - Section 19.180
#    Do autoencoders clean Gaussians ?

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [4]:
# %% Generate data

# 2D Gaussian params
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Vary FWHM smoothly
widths = np.linspace(2,20,n_gaussians)

# Preallocate tensors for images (N,channels,size,size)
images = torch.zeros(n_gaussians,1,img_size,img_size)

# Generate images
for i in range(n_gaussians):

    # Gaussian with random center offset
    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    # Layer some noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Add random bar randomly
    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    # Add to tensor
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(n_gaussians)
    G   = np.squeeze( images[pic,:,:] )

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.suptitle('Occluded Gaussians')

plt.savefig('figure47_autoencoder_clean_gaussian.png')
plt.show()
files.download('figure47_autoencoder_clean_gaussian.png')


In [8]:
# %% Function to generate the model

# Basically for the decoder we do "transpose" convolution
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2) )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()
yHat = CNN(images[:10,:,:,:])

print()
print(yHat.shape)

# Plotting
fig,ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(torch.squeeze(images[0,0,:,:]).detach(),cmap='jet')
ax[0].set_title('Model input')
ax[1].imshow(torch.squeeze(yHat[0,0,:,:]).detach(),cmap='jet')
ax[1].set_title('Model output')

plt.suptitle('Model test')
plt.tight_layout()

plt.savefig('figure48_autoencoder_clean_gaussian.png')
plt.show()
files.download('figure48_autoencoder_clean_gaussian.png')


In [ ]:
# %% Check all the parameters in the model

summary(CNN,(1,img_size,img_size))


In [14]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 500
    CNN,loss_fun,optimizer = gen_model()

    losses = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # pick a set of images at random
        pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
        X = images[pics_batch,:,:,:]

        # Forward propagation and loss
        yHat = CNN(X)
        loss = loss_fun(yHat,X)

        losses.append(loss.item())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return losses,CNN


In [48]:
# %% Run the model

losses,CNN = train_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(losses,'-',label='Train')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss (final loss=%.3f)'%losses[-1])

plt.savefig('figure49_autoencoder_clean_gaussian.png')
plt.show()
files.download('figure49_autoencoder_clean_gaussian.png')


In [ ]:
# %% Plotting

pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
X = images[pics_batch,:,:,:]
yHat = CNN(X)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(1.5*phi*6,6))

for i in range(10):

    G = torch.squeeze( X[i,0,:,:] ).detach()
    O = torch.squeeze( yHat[i,0,:,:] ).detach()

    axs[0,i].imshow(G,vmin=-1,vmax=1,cmap='jet')
    axs[0,i].axis('off')
    axs[0,i].set_title('Model input')

    axs[1,i].imshow(O,vmin=-1,vmax=1,cmap='jet')
    axs[1,i].axis('off')
    axs[1,i].set_title('Model output')

plt.tight_layout()

plt.savefig('figure50_autoencoder_clean_gaussian.png')
plt.show()
files.download('figure50_autoencoder_clean_gaussian.png')


In [54]:
# %% Exercise 1
#    There are no test data here, so how do you know whether the model overfit the training set? Fortunately, you can
#    simply create as much new data as you want! That's one of the advantages of generating data ;)
#    Generate a new dataset to use as a test set. How does the MSE loss compare on the test set? Did we overfit here?

# Seems to work quite well also on test data

# Generate more data
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

widths      = np.linspace(2,20,n_gaussians)
images_test = torch.zeros(n_gaussians,1,img_size,img_size)

for i in range(n_gaussians):

    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    G  = G + np.random.randn(img_size,img_size)/5

    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    images_test[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)

# Pass through model
pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
X = images_test[pics_batch,:,:,:]
yHat = CNN(X)


In [ ]:
# %% Exercise 1
#    Continue...

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(1.5*phi*6,6))

for i in range(10):

    G = torch.squeeze( X[i,0,:,:] ).detach()
    O = torch.squeeze( yHat[i,0,:,:] ).detach()

    axs[0,i].imshow(G,vmin=-1,vmax=1,cmap='jet')
    axs[0,i].axis('off')
    axs[0,i].set_title('Model input')

    axs[1,i].imshow(O,vmin=-1,vmax=1,cmap='jet')
    axs[1,i].axis('off')
    axs[1,i].set_title('Model output')

plt.tight_layout()

plt.savefig('figure51_autoencoder_clean_gaussian_extra1.png')
plt.show()
files.download('figure51_autoencoder_clean_gaussian_extra1.png')


In [25]:
# %% Exercise 2
#    The code here uses MaxPool. Are the results noticeably different if you use AvgPool instead?

# Roughly the same, possibly a bit smoother with mean pooling

# Function to generate the model
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.AvgPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.AvgPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2) )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [36]:
# %% Exercise 3
#    The final output doesn't have a nonlinearity (e.g., relu, sigmoid, tanh). Does it need one? Would that help? What
#    are some arguments for and against having a nonlinear activation function on the output of the decoder? Try adding
#    one and see if it improves (via the final loss and visual inspection) the result.

# ReLU yelds catastrophic effects, sigmoid yelds results as good as the standard
# model; must say that I cannot see exactly why; sure, the sigmoid is not
# clipping the values at zero like ReLU, but that said, why such a sharp
# difference ?

# Function to generate the model
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2),
                                          nn.Sigmoid() )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [47]:
# %% Exercise 4
#    I mentioned in the lecture "What are autoencoders and what do they do?" (section "Autoencoders") that BCEloss can be
#    used when the data are between 0 and 1. Normalize the images so they are appropriately scaled, and then use BCEloss
#    instead of MSEloss (does anything else in the model architecture need to change?). Which loss function gives a
#    better result?

# Well the data are continous so a MSELoss would be more appropriate and
# effective; a BCELoss would maybe work better for binary maps like masks,
# indeed the images retrieved with BCELoss look more like Gaussian masks that
# separate the centre of the distribution from the perifery

# Generate data
# 2D Gaussian params
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Vary FWHM smoothly
widths = np.linspace(2,20,n_gaussians)

# Preallocate tensors for images (N,channels,size,size)
images = torch.zeros(n_gaussians,1,img_size,img_size)

# Generate images
for i in range(n_gaussians):

    # Gaussian with random center offset
    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    # Layer some noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Add random bar randomly
    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    # Normalise data
    G = (G - np.min(G)) / (np.max(G) - np.min(G))

    # Add to tensor
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


# Function to generate the model
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2) )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.BCEWithLogitsLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer
